# Playfit Intelligence Lab — 02: Feature Engineering

Construcción de la matriz de features para el motor de recomendación.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

from src.features.game_features import build_feature_matrix, compute_popularity_score, compute_richness_score
from src.models.content_based import build_content_model

sns.set_theme(style="darkgrid")
print("Librerías cargadas.")

## 1. Feature Matrix
Construimos la matriz con tags (one-hot), plataformas (one-hot), scores, ventas, sentimiento y confianza.

In [ ]:
fm = build_feature_matrix()
fm = compute_popularity_score(fm)
fm = compute_richness_score(fm)
print(f"Feature matrix: {fm.shape[0]:,} juegos × {fm.shape[1]} columnas")
print(f"\nColumnas por tipo:")
meta = {'game_id', 'title', 'release_year', 'genre_id', 'best_critic_score', 'best_user_score',
        'critic_review_count', 'user_review_count', 'critic_positive_ratio', 'user_positive_ratio',
        'max_global_sales_millions', 'total_global_sales_millions', 'data_confidence_score',
        'has_external_id', 'has_company', 'has_age_rating', 'has_summary', 'has_sales',
        'has_review_sentiment', 'log_sales', 'total_review_count', 'popularity_score', 'richness_score'}
content_cols = [c for c in fm.columns if c not in meta]
print(f"  Tags/Platforms (binary): {len(content_cols)}")
print(f"  Numéricas (scores, sales, etc.): {len(meta) - 2}")

## 2. Análisis de Features
### 2.1 Tags más comunes

In [ ]:
tag_counts = {}
for col in content_cols:
    if col not in [c for c in fm.columns if c.startswith(('ps', 'nintendo', 'xbox', 'pc', 'sega', 'atari', 'arcade', 'switch', 'gamecube', 'game_gear', 'dreamcast', 'saturn', 'genesis', 'snes', 'nes', 'gb', 'gbc', 'gba', 'n64', 'ds', '3ds', 'psp', 'ps_vita', 'wii', 'wii_u', 'neo_geo'))]:
        tag_counts[col] = fm[col].sum()

sorted_tags = sorted(tag_counts.items(), key=lambda x: x[1], reverse=True)
print("Top 20 tags más comunes:")
for tag, count in sorted_tags[:20]:
    print(f"  {tag:25s}: {int(count):>6,} juegos")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
tags_names, tags_counts = zip(*sorted_tags[:20])
ax.barh(range(len(tags_names)), tags_counts, color='#3498db')
ax.set_yticks(range(len(tags_names)))
ax.set_yticklabels(tags_names)
ax.set_xlabel('Juegos')
ax.set_title('Top 20 Tags más comunes')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('../reports/figures/top_tags.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Distribución de Popularity Score

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.hist(fm['popularity_score'].to_numpy(), bins=50, color='#2ecc71', edgecolor='white')
ax1.set_title('Distribución de Popularity Score')
ax1.set_xlabel('Popularity Score')

ax2.scatter(
    fm['popularity_score'].to_numpy(),
    fm['data_confidence_score'].to_numpy(),
    alpha=0.1, s=1, color='#3498db'
)
ax2.set_title('Popularity vs Data Confidence')
ax2.set_xlabel('Popularity Score')
ax2.set_ylabel('Confidence Score')

plt.tight_layout()
plt.savefig('../reports/figures/popularity_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.3 Reducción de dimensionalidad (SVD)

In [ ]:
cm = build_content_model(fm, n_components=100)
reduced = cm['reduced']
print(f"Dimensión original: {len(content_cols)}")
print(f"Dimensión reducida: {reduced.shape[1]}")
print(f"Varianza explicada (primeros componentes): {cm['svd'].explained_variance_ratio_[:5]}")
print(f"Varianza explicada acumulada (100 comps): {cm['svd'].explained_variance_ratio_.sum():.4f}")

## 3. Resumen de Features Creadas

In [ ]:
print("\nFeatures disponibles para el modelo:")
print(f"  - {len(content_cols)} features de contenido (tags + plataformas)")
print(f"  - Popularity score compuesto (scores + ventas + confianza)")
print(f"  - Richness score (cantidad de datos disponibles)")
print(f"  - Data confidence score (calidad de datos: 0-100)")
print(f"  - Embedding SVD-100 (representación densa de contenido)")
print(f"  - Features de señal: critic/user score, review counts, sales, sentiment")
print(f"\nMatriz guardada en: data/processed/feature_matrix.parquet ({fm.shape[0]:,} × {fm.shape[1]})")